In [11]:
from SteerEnergyStorage.Formulations.ElectrodeFormulations import ElectrodeFormulation
from SteerEnergyStorage.Constructions.Electrodes import Cathode, Anode
from SteerEnergyStorage.Formulations.Stacks import Stack
from SteerEnergyStorage.Constructions.Cells import StackedPouchCell
from SteerEnergyStorage.Materials.ElectrodeMaterials import ActiveMaterial, Binder, ConductiveAdditive
from SteerEnergyStorage.Materials.CurrentCollectors import CurrentCollector
from SteerEnergyStorage.Materials.Separators import Separator
from SteerEnergyStorage.Materials.Electrolytes import Electrolyte
from SteerEnergyStorage.Constructions.Containers import Pouch
from SteerEnergyStorage.Materials.other import Laminate, Tape, Terminal

In [12]:
# construct cathode
cathode_active_material = ActiveMaterial(name="Faradion_Gen2_4.25V", 
                                            formula="Li2MnSiO4", 
                                            specific_cost=11.26, 
                                            density=4, 
                                            irreversible_capacity_scaling=1, 
                                            reversible_capacity_scaling=1,
                                            half_cell_path='../Data/Cathode_Faradion_Gen2_4.25V.csv')

cathode_conductive_additive = ConductiveAdditive(specific_cost=9, density=1.9)

cathode_binder = Binder(name="PVDF", specific_cost=15, density=1.7)

cathode_formulation = ElectrodeFormulation(active_materials={cathode_active_material: 89},
                                            binder={cathode_binder: 5},
                                            conductive_additive={cathode_conductive_additive: 6})

cathode_current_collector = CurrentCollector(name="Aluminium", 
                                             formula="Al", 
                                             specific_cost=6.30, 
                                             density=2.7, 
                                             thickness=15, 
                                             length=16.0,
                                             width=10.8,
                                             bare_tab_area=8.22)

cathode = Cathode(formulation=cathode_formulation,
                    mass_loading=10.68,
                    current_collector=cathode_current_collector,
                    swell_factor=1.0,
                    calender_density=2.60)


In [13]:

# construct anode
anode_active_material = ActiveMaterial(name="Faradion_HC",
                                        formula="Na2Ti3O7",
                                        specific_cost=14.27,
                                        density=1.50,
                                        irreversible_capacity_scaling=1,
                                        reversible_capacity_scaling=1,
                                        half_cell_path='../Data/Anode_Faradion_HC.csv')

anode_conductive_additive = ConductiveAdditive(specific_cost=9, density=1.9)

anode_binder = Binder(name="PVDF", specific_cost=10, density=1.7)

anode_formulation = ElectrodeFormulation(active_materials={anode_active_material: 88},
                                            binder={anode_binder: 3},
                                            conductive_additive={anode_conductive_additive: 9})

anode_current_collector = CurrentCollector(name="Copper",
                                            formula="Cu",
                                            specific_cost=6.30,
                                            density=2.70,
                                            thickness=15,
                                            length=16.0,
                                            width=10.8,
                                            bare_tab_area=7.55)

anode = Anode(formulation=anode_formulation,
                mass_loading=5.25,
                current_collector=anode_current_collector,
                swell_factor=1.0,
                calender_density=0.85)


In [14]:

# construct seperator
seperator = Separator(thickness=16, 
                      areal_cost=0.9, 
                      density=0.4, 
                      slit_width=100, 
                      porosity=47, 
                      fold_length=186)

# construct the stack
stack = Stack(anode=anode, 
                cathode=cathode,
                seperator=seperator, 
                n_p_ratio=1.13,
                n_stacks=26)


In [15]:

# make electrolyte
electrolyte = Electrolyte(name="LiPF6", 
                            formula="LiPF6", 
                            specific_cost=8.94, 
                            density=1.2)

# make the pouch
laminate = Laminate(thickness=113, areal_mass=18, areal_cost=4.64)
tape = Tape(mass=0.3)

pouch = Pouch(laminate=laminate, 
                heat_seal_size_sides=7, 
                heat_seal_size_top=22, 
                tape=tape)

# Make the cell
pos_terminal = Terminal(mass = 1, specific_cost = 16, name="Positive Terminal")
neg_terminal = Terminal(mass = 1, specific_cost = 16, name="Negative Terminal")

cell = StackedPouchCell(pouch=pouch,
                        stack=stack,
                        electrolyte=electrolyte,
                        electrolyte_overfill=10,
                        voltage_upper_cut_off=4.2,
                        voltage_lower_cut_off=1.0,
                        positive_terminal=pos_terminal,
                        reversible_capacity=11934,
                        irreversible_capacity=1215,
                        grid_n=200,
                        negative_terminal=neg_terminal)

In [16]:
cell.cost_breakdown

{stack: 2.92,
 Pouch: 0.103,
 LiPF6: 0.509,
 Positive Terminal: 0.016,
 Negative Terminal: 0.016}

In [17]:
cell.stack.cost_breakdown

{Cathode: 1.206, Anode: 0.794, Separator: 0.921}

In [18]:
cell.stack.cathode_cost_breakdown

{'Active Material': {Faradion_Gen2_4.25V: 0.962},
 'Binder': {PVDF: 0.072},
 'Conductive Additive': {conductive additive: 0.052},
 'Current Collector': 0.12}

In [19]:
cell.stack.anode_cost_breakdown

{'Active Materials': {Faradion_HC: 0.615},
 'Binders': {PVDF: 0.015},
 'Conductive Additives': {conductive additive: 0.04},
 'Current Collector': 0.124}

In [21]:
figure = cell.get_capacity_voltage_plot(width=900, height=600)
figure.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(
    data = [
        go.Sankey(
            node=dict(
                pad=15,
                thickness=10,
                line=dict(color="black", width=1.5),
                label=["cell", 
                       "stack", "pouch", "electrolyte", "positive terminal", "negative terminal", 
                       "cathode", "anode", "seperator",
                       "active materials", "binders", "conductive additives", "current collector",
                       "active materials", "binders", "conductive additives", "current collector"],
                color="black",
                x=[0, 
                   0.25, 0.25, 0.25, 0.25, 0.25, 
                   0.5, 0.5, 0.5,
                   0.75, 0.75, 0.75, 0.75,
                   0.75, 0.75, 0.75, 0.75],
                y=[0,
                   0.3, 0.5, 0.6, 0.7, 0.8,
                   0.3, 0.5, 0.7,
                   0.1, 0.2, 0.3, 0.4,
                   0.6, 0.7, 0.8, 0.9]
            ),
            link=dict(
                source = [0, 0, 0, 0, 0, 1, 1, 1, 6, 6, 6, 6, 7, 7, 7, 7],
                target = [1, 2, 3, 4, 5, 6, 7, 8, 9,10,11,12,13,14,15,16],
                value = [cell.cost_breakdown[cell.stack], 
                         cell.cost_breakdown[cell.pouch], 
                         cell.cost_breakdown[cell.electrolyte], 
                         cell.cost_breakdown[cell.positive_terminal], 
                         cell.cost_breakdown[cell.negative_terminal],
                         cell.stack.cost_breakdown[cell.stack.cathode],
                         cell.stack.cost_breakdown[cell.stack.anode],
                         cell.stack.cost_breakdown[cell.stack.seperator],
                         sum(cell.stack.cathode_cost_breakdown['Active Material'].values()),
                         sum(cell.stack.cathode_cost_breakdown['Binder'].values()),
                         sum(cell.stack.cathode_cost_breakdown['Conductive Additive'].values()),
                         cell.stack.cathode_cost_breakdown['Current Collector'],
                         sum(cell.stack.anode_cost_breakdown['Active Materials'].values()),
                         sum(cell.stack.anode_cost_breakdown['Binders'].values()),
                         sum(cell.stack.anode_cost_breakdown['Conductive Additives'].values()),
                         cell.stack.anode_cost_breakdown['Current Collector']
                         ]
            )
        )
    ]
)

fig.show()
